# Section 1: Raw Data Inspection and Cleaning (House 1)
This notebook walks through the inspection, data-quality profiling, cleaning,
and 1-minute resampling of raw REFIT data for House 1.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, JSON

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_house1
from src.cleaning import inspect_raw_data, clean_raw_data
from src.resampling import resample_to_1min
from src.visualization import (
    plot_raw_vs_cleaned,
    plot_missing_data_heatmap,
    plot_sampling_intervals
)


## 1. Load Raw House 1 Data
House 1 data is supplied in two parts: Part1 and Part2. We load and concatenate them chronologically.


In [ ]:
print('Loading raw House 1 data...')
df_raw = load_raw_house1()
print(f'Raw data shape: {df_raw.shape}')
df_raw.head()


## 2. Inspect Raw Data Quality
We identify missing values, duplicate timestamps, non-monotonic records, sampling interval distribution, and extreme spikes.


In [ ]:
quality_report = inspect_raw_data(df_raw, house_id=1)
print('=== Key Quality Findings ===')
print(f'Total Rows: {quality_report["total_rows"]:,}')
print(f'Duplicate Timestamps: {quality_report["duplicate_timestamps"]:,}')
print(f'Sampling Cadence: mean={quality_report["sampling_interval_seconds"]["mean"]:.2f}s, '
      f'median={quality_report["sampling_interval_seconds"]["median"]:.1f}s')
print(f'Aggregate Extreme Spikes (>4000 W): {quality_report["extreme_readings_gt_4000w"]["Aggregate"]:,}')
print(f'Major Outages (>60s): {quality_report["significant_gaps_gt_60s_count"]:,}')


## 3. Clean Raw Data
We apply reproducible cleaning rules:
1. Deduplicate timestamps by retaining the latest reading.
2. Clip negative power readings to 0 W.
3. Replace impossible power spikes (>4000 W) with NaN to preserve time index.
4. Sort chronologically.


In [ ]:
df_clean, cleaning_stats = clean_raw_data(df_raw)
print('Cleaning Stats:', cleaning_stats)
print(f'Rows before: {cleaning_stats["rows_before"]:,} -> after: {cleaning_stats["rows_after"]:,}')


## 4. Resample to 1-Minute Resolution
1. Instantaneous power is averaged over 1-minute intervals (unit remains Watts).
2. Short sensor dropouts (<= 3 minutes) are conservatively interpolated.
3. Prolonged outages (> 15 minutes) are flagged and preserved as explicit missingness (never fabricated).


In [ ]:
df_1min, resamp_stats = resample_to_1min(df_clean, max_interp_gap_min=3, outage_gap_min=15)
print('Resampling Stats:', resamp_stats)
df_1min.head()


## 5. Visualizations
Reviewing before/after plots, sampling interval distribution, and outage heatmaps.


In [ ]:
# Plot sampling intervals
from PIL import Image
Image.open(PROJECT_ROOT / 'outputs/figures/section1_cleaning/house1_sampling_intervals.png')


In [ ]:
# Plot Aggregate Raw vs Cleaned
Image.open(PROJECT_ROOT / 'outputs/figures/section1_cleaning/house1_aggregate_raw_vs_cleaned.png')


In [ ]:
# Plot Missing Data Heatmap
Image.open(PROJECT_ROOT / 'outputs/figures/section1_cleaning/house1_missing_data_heatmap.png')
